In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


In [ ]:
pd.set_option("display.max_columns",None)
pd.set_option("display.max_rows",None)

Employee Attrition Prediction:
Problem Statement: Predict whether an employee leaves the company or not(attrition of the employee)  by understanding the underlying patterns that are associated with the attrition using the provided dataset.
(It is a Binary Classification Task where the output is (Yes/No) type)

Objective: 
The main objective of this project is to build a machine learning model to predict employee attrition using employee-related features, helping the HR department identify patterns associated with attrition and support workforce planning and appropriate retention strategies.

1.Load the dataset

In [ ]:
data=pd.read_csv(r"D:\ml_project(assignment_mlops)\data\raw\WA_Fn-UseC_-HR-Employee-Attrition.csv")

Understand the dataset structure

In [ ]:
data.shape


In [ ]:
data.size

In [ ]:
data.head()

In [ ]:
data.info()

In [ ]:
data.head()

Know how many categorical columns in dataset

In [ ]:
data.select_dtypes(include="object").shape[1]

so we need to convert these 9 columns to numeric form (encode) since model works only on numeric data. We may also not include them in training process if they are not necessary(Ex:An identifier column)

know how many numerical columns are present in dataset

In [ ]:
data.select_dtypes(include="number").shape[1]

DATA VALIDATION

Missing values

In [ ]:
data.isnull().sum()

overall count of missing values

In [ ]:
data.isnull().sum().sum()

Finding the columns that are categorical and storing their names as a list

In [ ]:
obj_col=data.select_dtypes(include="object").columns

Sometimes .isna() might show 0 missing values but categorical columns might have empty strings or whitespaces which are also considered as missing values. So find the count of these empty strings/whitespaces

In [ ]:
(data[obj_col].apply(lambda col:col.str.strip()=="")).sum()

Stastical summary of numerical columns(Data Validation-(see min and max values if they have valid ranges or not))

In [ ]:
#missing_percentage=(data.isnull().mean())*100
#Is the missingness random?
#Why might they be missing? here i dont have any missing values

In [ ]:
data.describe()

The numerical features were inspected using descriptive statistics. No obviously invalid negative values or extreme ranges were observed based on the expected meaning of the variables

Know the categories of each categorical column

In [ ]:
for col in obj_col:
    print(f"{col}:{list(data[col].unique())}")

A feature with only one unique value provides no information for distinguishing one employee from another.
there's nothing useful to learn from that feature.

This is called a constant feature or a zero-variance feature.
so we would not consider this over18 column in our feature set

know how many samples belong to a particular category- value_counts()

tells us the distribution of that categorical feature

In [ ]:
for col in obj_col:
    print(f"{data[col].value_counts()}")
    print()

Checking Duplicates

In [ ]:
data.duplicated().sum()

Data Validation Summary: The dataset contains 1,470 employee records and 35 columns. No null values, empty strings, or duplicate records were detected. Numerical features did not show obviously invalid ranges, and categorical variables contained consistent values. The Over18 feature was identified as a constant feature because all observations have the value Y, making it a potential candidate for removal during feature selection.

EDA

In [ ]:
data.nunique(axis=0)

EmployeeCount,Over18,StandardHours are having only 1 value. So they are useless as variance is 0.

In [ ]:
data.select_dtypes(include="object").apply(lambda x:x.str.strip().eq("").sum())

In [ ]:
#x.str.strip() targets cells: The .str prefix tells pandas to look inside the Series container and apply the strip operation to every individual cell element-by-element.

#.eq("") creates a Boolean mask: It checks every cell individually. It yields a Series of True or False values matching the original row length (e.g., [False, True, False]).
#x.str.strip().eq("") outputs a pandas Series:Pandas objects are highly specialized. When you perform an operation using .str or .eq(), pandas keeps the data wrapped inside a Series container (looks like a column with index numbers).Pandas Series do have a .sum() method:Unlike a standard Python list, the pandas library explicitly built a .sum() method directly into the Series object.
#When you call .sum() on a pandas boolean Series, pandas automatically treats True as 1 and False as 0 behind the scenes, giving you the correct total count.

In [ ]:
numerical_cols = [
    'Age','DailyRate','DistanceFromHome','HourlyRate',
    'MonthlyIncome','MonthlyRate','NumCompaniesWorked',
    'PercentSalaryHike','TotalWorkingYears',
    'TrainingTimesLastYear','YearsAtCompany',
    'YearsInCurrentRole','YearsSinceLastPromotion',
    'YearsWithCurrManager']
data[numerical_cols].describe().T

In [ ]:
potential_outliers=[
"MonthlyIncome",
"NumCompaniesWorked",	
"TotalWorkingYears","YearsAtCompany","YearsInCurrentRole",
"YearsSinceLastPromotion","YearsWithCurrManager"]

In [ ]:
for col in potential_outliers:
    plt.figure()
    sns.boxplot(data[col])
    plt.title(col)
    plt.show()
    


all of the assumed columns have outliers. Let us plot for all remaining numeric columns this time

In [ ]:
for col in numerical_cols:
    if col not in potential_outliers:
        plt.figure()
        sns.boxplot(data[col])
        plt.title(col)
        plt.show()

So from the above box plots, TrainingTimesLastYear, MonthlyIncome,
NumCompaniesWorked,	 TotalWorkingYears, YearsAtCompany,YearsInCurrentRole, YearsSinceLastPromotion,YearsWithCurrManager are having outliers

Understanding the distribution of categorical features

In [ ]:
for col in obj_col:
    print("\n")
    print(data[col].value_counts(normalize=True)*100)

In [ ]:
ordinal_cols = [
    'Education',
    'EnvironmentSatisfaction',
    'JobInvolvement',
    'JobLevel',
    'JobSatisfaction',
    'RelationshipSatisfaction',
    'StockOptionLevel',
    'WorkLifeBalance',
    'PerformanceRating'
]

Understanding the distribution of ordinal features

In [ ]:
for col in ordinal_cols:
    print("\n")
    print((data[col].value_counts(normalize=True)*100).sort_index())

Analysing if target class is balanced/imbalanced

In [ ]:
data["Attrition"].value_counts(normalize=True)*100

In [ ]:
plt.figure()
sns.countplot(data=data,x="Attrition")
plt.title("Attrition")
plt.show()

Around 16.1% of employees are associated with Attrition where as 83.8% employees stayed.

#categorical columns vs target analysis

In [ ]:
pd.crosstab(data["OverTime"],data["Attrition"],normalize=True)*100

In [ ]:
for col in obj_col:
    if col=="Attrition":
        continue
    print()
    print(pd.crosstab(data[col],data["Attrition"],normalize=True)["Yes"]*100)

The categories "Travel_rarely, R&D, Lifesciences, Male, Laboratory Technician, Single(Marital Status), who does overtime" have been associated with higher observed attrition rates than the other categories of same respective features

ordinal columns vs target analysis

In [ ]:
for col in ordinal_cols:
    
    print()
    print(pd.crosstab(data[col],data["Attrition"],normalize=True)["Yes"]*100)

JobLevel,StockOptionLevel,Performance Rating are showing a monotonic relationship with target


In [ ]:
from sklearn.preprocessing import LabelEncoder
le=LabelEncoder()
data["Attrition"]=le.fit_transform(data["Attrition"])
data.head()



In [ ]:
corr_data=data[numerical_cols]
corr_data["Attrition"]=data["Attrition"]
corr=corr_data.corr()
plt.figure(figsize=(15,10))

sns.heatmap(corr,annot=True,fmt='.2f')
plt.show()

Data Preprocessing and Feature selection

In [ ]:
data.columns

In [ ]:
x=data.drop(columns=["Attrition","EmployeeCount","EmployeeNumber","Over18","StandardHours"])


In [ ]:
y=data["Attrition"]

In [ ]:
from sklearn.model_selection import train_test_split

x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,stratify=y,random_state=42,shuffle=True)
#shuffle helps to generalize the model but you should not shuffle in case of time series and when the order of samples matter


In [ ]:
cols=[]
for col in obj_col:
    if col in x_train.columns:
        cols.append(col)

In [ ]:
from sklearn.preprocessing import OneHotEncoder
oe=OneHotEncoder(drop='first',handle_unknown='ignore',sparse_output=False)

oe.fit(x_train[cols])
x_train_encoded=oe.transform(x_train[cols])
x_test_encoded=oe.transform(x_test[cols])






In [ ]:
df_train_encode=pd.DataFrame(x_train_encoded,columns=oe.get_feature_names_out(),index=x_train.index)
df_test_encode=pd.DataFrame(x_test_encoded,columns=oe.get_feature_names_out(),index=x_test.index)
x_train_encode=pd.concat([x_train.drop(columns=cols),df_train_encode],axis=1)
x_test_encode=pd.concat([x_test.drop(columns=cols),df_test_encode],axis=1)


In [ ]:
from sklearn.preprocessing import StandardScaler
scaler=StandardScaler()
x_train_scaled=x_train_encode.copy()
x_test_scaled=x_test_encode.copy()
x_train_scaled[numerical_cols]=scaler.fit_transform(x_train_encode[numerical_cols])
x_test_scaled[numerical_cols]=scaler.transform(x_test_encode[numerical_cols])

In [ ]:
print(x_train_encode.shape,x_test.shape)
print(x_train_encode.dtypes.value_counts())

here we can observe an increase in number of columns

In [ ]:
x_train_scaled.head()

Now let us see if there are any missing values

In [ ]:
print(x_train_scaled.isnull().sum().sum(),
x_test_scaled.isnull().sum().sum())

Training Models

In [ ]:
#Baseline Model
from sklearn.linear_model import LogisticRegression
lr=LogisticRegression(max_iter=1000)
lr.fit(x_train_scaled,y_train)
y_pred_lr=lr.predict(x_test_scaled)
from sklearn.metrics import confusion_matrix,classification_report
cm=confusion_matrix(y_test,y_pred_lr)
plt.figure(figsize=(10,6))
sns.heatmap(cm,annot=True,fmt='d',cmap="Blues",xticklabels=["No","Yes"],yticklabels=["No","Yes"])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()
print()
print(classification_report(y_test,y_pred_lr))

 a small note:since we passed (y_actual,y_pred) row wise(y_axis) we have actual and column wise(x_axis) we have predicted
 here recall is low for class 1(Yes) i.e only 34%.
 overall the metrics are:
 Accuracy: 86%
 Precision(without considering sample size for classes):76%
 Recall: 65%
 F1 score: 68%
 We need to focus on improving recall score since we need to concentrate on FalseNegative i.e model predicts they are not leaving but actually they are leaving the company
Since our baseline model is showing low recall for minority class, we need to explore class balancing techniques

Note:class_weight="balanced" tells Logistic Regression:

"Don't treat the minority class as less important just because there are fewer examples." The model gives more importance to the minority class.
So the model becomes more willing to predict Yes.

In [ ]:
lr_balanced=LogisticRegression(max_iter=1000,class_weight="balanced")
lr_balanced.fit(x_train_scaled,y_train)
y_pred_lr_balanced=lr_balanced.predict(x_test_scaled)
print(classification_report(y_test,y_pred_lr_balanced))

Now the recall score improved but precision score has decreased(Tradeoff)
Some actual No -> predicted Yes->False Positives increased and then Precision decreased

Decision Tree(Baseline)

In [ ]:
from sklearn.tree import DecisionTreeClassifier
dt=DecisionTreeClassifier(random_state=42) #random_state for consistent results and reproducibility
dt.fit(x_train_scaled,y_train)
y_pred_dt=dt.predict(x_test_scaled)
print(classification_report(y_test,y_pred_dt))

The default baseline model of decision tree (without any hyperparameter) have low recall, precision and f1 score.

RandomForest(Baseline)

In [ ]:
from sklearn.ensemble import RandomForestClassifier
rf=RandomForestClassifier(random_state=42)
rf.fit(x_train_scaled,y_train)
y_pred_rf=rf.predict(x_test_scaled)
print(classification_report(y_test,y_pred_rf))


svm baseline model

In [ ]:
from sklearn.svm import SVC
svm=SVC(random_state=42)
svm.fit(x_train_scaled,y_train)
y_pred_svm=svm.predict(x_test_scaled)
print(classification_report(y_test,y_pred_svm))

Balanced SVM

In [ ]:
svm_balanced=SVC(random_state=42,class_weight="balanced")
svm_balanced.fit(x_train_scaled,y_train)
y_pred_svm_balanced=svm_balanced.predict(x_test_scaled)
print(classification_report(y_test,y_pred_svm_balanced))

we observe one thing that class imbalance is affecting the metrics in a bad way and when we take class_weight=balanced where models tries to balance the total impact of each class during training. both svm, logistic regression provided better metric scores than their baseline models
So handling class imbalance might actually imporove the metrics

Now let us see another technique to handle class imbalance- Smote

In [ ]:
print(y_train.value_counts())

In [ ]:
from imblearn.over_sampling import SMOTE
smote=SMOTE(random_state=42)
x_train_smote,y_train_smote=smote.fit_resample(x_train_scaled,y_train)
print(y_train_smote.value_counts())

In [ ]:
lr_smote=LogisticRegression(max_iter=100,random_state=42)
lr_smote.fit(x_train_smote,y_train_smote)
y_pred_lr_smote=lr_smote.predict(x_test_scaled)
print(classification_report(y_test,y_pred_lr_smote))

#training svm on smote data

In [ ]:
from sklearn.svm import SVC
svm_smote=SVC(random_state=42)
svm_smote.fit(x_train_smote,y_train_smote)
y_pred_svm_smote=svm.predict(x_test_scaled)
print(classification_report(y_test,y_pred_svm_smote))

here smote did not help to improve metrics. for SVM, class weighting clearly helped much more than SMOTE

In [ ]:
from sklearn.tree import DecisionTreeClassifier
dt_smote=DecisionTreeClassifier(random_state=42) #random_state for consistent results and reproducibility
dt_smote.fit(x_train_smote,y_train_smote)
y_pred_dt_smote=dt.predict(x_test_scaled)
print(classification_report(y_test,y_pred_dt_smote))

Decision tree with smote:
Metrics
Accuracy-76
precision(overall)-59 and for class1- 31
recall(overall)-61 and for class1-38
f1 score(overall)-60 and for class1-34

random forest+smote


In [ ]:
from sklearn.ensemble import RandomForestClassifier
rf_smote=RandomForestClassifier(random_state=42)
rf_smote.fit(x_train_smote,y_train_smote)
y_pred_rf_smote=rf.predict(x_test_scaled)
print(classification_report(y_test,y_pred_rf_smote))

Random forest +smote (metrics)
Accuracy-83
precision(overall)-63 and for class1- 42
recall(overall)-54 and for class1-11
f1 score(overall)-54 and for class1-17

#cross validation
cross validation helps us to evaluate whether the results produced by model are consistent across various test folds

Till now the best models are baseline logistic regression, balanced logistic regression and balanced svm
Now let us see how consistent their performanace is using cross validation

In [ ]:
from sklearn.model_selection import StratifiedKFold

skf = StratifiedKFold(n_splits=5,shuffle=True,random_state=42)
from sklearn.model_selection import cross_validate
scoring={"accuracy":"accuracy","precision":"precision","recall":"recall","f1":"f1"}
scores_lr=cross_validate(lr,x_train_scaled,y_train,scoring=scoring,cv=skf)
for metric in scoring:
    res=scores_lr[f"test_{metric}"]
    print(f"Metric:{metric}")
    print(f"Scores:{res}")
    print(f"Mean score:{res.mean()}")
    print(f"SD of score:{res.std()}")
    


overall f1 score is 68% for logistic regression baseline model and on cross validation, it's overall f1 score is 56%

In [ ]:
scores_lr_balanced=cross_validate(lr_balanced,x_train_scaled,y_train,cv=skf,scoring=scoring)
for metric in scoring:
    res=scores_lr_balanced[f"test_{metric}"]
    print()
    print(f"Metric:{metric}")
    print(f"Scores:{res}")
    print(f"Mean score:{res.mean()}")
    print(f"SD of score:{res.std()}")

In [ ]:
scores_svm_balanced=cross_validate(svm_balanced,x_train_scaled,y_train,cv=skf,scoring=scoring)
for metric in scoring:
    res=scores_svm_balanced[f"test_{metric}"]
    print()
    print(f"Metric:{metric}")
    print(f"Scores:{res}")
    print(f"Mean score:{res.mean()}")
    print(f"SD of score:{res.std()}")

here precision,recall,f1score are calculated only for class1(Attrition=Yes) which is what we want to focus mainly
On overall balanced logistic regression, balanced svm leads in these scores
But balanced svm has a better f1 score of 54% whereas balanced logistic regression has 48%of f1 score
Balanced Logistic regression has better recall of 73% whereas balanced_svm has recall of 69%

Let us tune balanced sv, balanced lr models as they are performing better. 

c controls strength of regularization
small c-> stronger regularization
large c-> weaker regularization 
regularization means how well the model can fit the data

In [ ]:
from sklearn.model_selection import GridSearchCV
param_grid = {"C": [0.01, 0.1, 1, 10, 100]}
grid_lr = GridSearchCV(estimator=lr_balanced,param_grid=param_grid,cv=skf,scoring="f1",n_jobs=-1)
grid_lr.fit(x_train_scaled,y_train)

In [ ]:
print(grid_lr.best_params_)
print(grid_lr.best_score_)

In [ ]:
from sklearn.model_selection import GridSearchCV
param_grid = {"C": [0.01, 0.1, 1, 10, 100]}
grid_svm = GridSearchCV(estimator=svm_balanced,param_grid=param_grid,cv=skf,scoring="f1",n_jobs=-1)
grid_svm.fit(x_train_scaled,y_train)
print(grid_svm.best_params_)
print(grid_svm.best_score_)

In [ ]:
best_lr=grid_lr.best_estimator_
best_svm=grid_svm.best_estimator_


In [ ]:
y_pred_lr_tuned = best_lr.predict(x_test_scaled)

y_pred_svm_tuned = best_svm.predict(x_test_scaled)


print("Balanced Logistic Regression - Tuned")
print(classification_report(y_test, y_pred_lr_tuned))

print("\nBalanced SVM - Tuned")
print(classification_report(y_test, y_pred_svm_tuned))

Balanced SVM was selected as the final model because it achieved higher accuracy, precision, and F1-score mainly for the minority class(Attrition=Yes), while maintaining a recall  very close to Balanced Logistic Regression .
SVM provides a better overall balance for predicting employee attrition.

#Prediction on unseen data

In [ ]:
new_employee = pd.DataFrame([{
    "Age": 29,
    "DailyRate": 800,
    "DistanceFromHome": 5,
    "Education": 3,
    "EnvironmentSatisfaction": 3,
    "HourlyRate": 70,
    "JobInvolvement": 3,
    "JobLevel": 2,
    "JobSatisfaction": 3,
    "MonthlyIncome": 5000,
    "MonthlyRate": 15000,
    "NumCompaniesWorked": 2,
    "PercentSalaryHike": 14,
    "PerformanceRating": 3,
    "RelationshipSatisfaction": 3,
    "StockOptionLevel": 1,
    "TotalWorkingYears": 7,
    "TrainingTimesLastYear": 3,
    "WorkLifeBalance": 3,
    "YearsAtCompany": 5,
    "YearsInCurrentRole": 3,
    "YearsSinceLastPromotion": 1,
    "YearsWithCurrManager": 3,

    "BusinessTravel": "Travel_Rarely",
    "Department": "Research & Development",
    "EducationField": "Life Sciences",
    "Gender": "Male",
    "JobRole": "Research Scientist",
    "MaritalStatus": "Single",
    "OverTime": "Yes"
}])
new_encoded = oe.transform(new_employee[cols])

new_encoded = pd.DataFrame(new_encoded,columns=oe.get_feature_names_out(cols),index=new_employee.index)
new_processed = pd.concat([new_employee.drop(columns=cols),new_encoded],axis=1)
print(new_processed.shape)

In [ ]:
new_processed[numerical_cols] = scaler.transform(new_processed[numerical_cols])
prediction = best_svm.predict(new_processed)

print("Prediction:", prediction)

In [ ]:
if prediction[0] == 1:
    print("Prediction: Employee is likely to leave")
else:
    print("Prediction: Employee is likely to stay")

In [ ]:
new_employee_2 = pd.DataFrame([{
    "Age": 42,
    "DailyRate": 1200,
    "DistanceFromHome": 2,
    "Education": 4,
    "EnvironmentSatisfaction": 4,
    "HourlyRate": 80,
    "JobInvolvement": 4,
    "JobLevel": 3,
    "JobSatisfaction": 4,
    "MonthlyIncome": 9000,
    "MonthlyRate": 18000,
    "NumCompaniesWorked": 1,
    "PercentSalaryHike": 18,
    "PerformanceRating": 4,
    "RelationshipSatisfaction": 4,
    "StockOptionLevel": 2,
    "TotalWorkingYears": 15,
    "TrainingTimesLastYear": 3,
    "WorkLifeBalance": 4,
    "YearsAtCompany": 10,
    "YearsInCurrentRole": 6,
    "YearsSinceLastPromotion": 3,
    "YearsWithCurrManager": 6,

    "BusinessTravel": "Travel_Rarely",
    "Department": "Research & Development",
    "EducationField": "Medical",
    "Gender": "Female",
    "JobRole": "Research Scientist",
    "MaritalStatus": "Married",
    "OverTime": "No"
}])
new_encoded_2 = oe.transform(new_employee_2[cols])

new_encoded_2 = pd.DataFrame(new_encoded_2,columns=oe.get_feature_names_out(cols),index=new_employee_2.index)

new_processed_2 = pd.concat( [new_employee_2.drop(columns=cols),new_encoded_2], axis=1)

new_processed_2[numerical_cols] = scaler.transform(new_processed_2[numerical_cols])
prediction_2 = best_svm.predict(new_processed_2)

print("Prediction:", prediction_2)

if prediction_2[0] == 1:
    print("Employee is likely to leave")
else:
    print("Employee is likely to stay")

##Conclusion

In this project, different machine learning models were developed and evaluated for employee attrition prediction. After preprocessing, handling class imbalance, cross-validation, and hyperparameter tuning, Balanced SVM with C = 1 was selected as the final model. It achieved 80% accuracy, 66% recall, and 51% F1-score for Class 1 (Attrition = Yes). The model was also tested on unseen employee data and successfully generated predictions.
